In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 136)


In [3]:
competition_features = [

    "customer_count",

    "website_traffic",

    "linkedin_followers",

    "twitter_followers",

    "media_mentions",

    "domain_authority"

]

for col in competition_features:

    df[f"{col}_norm"] = (

        df[col] - df[col].min()

    ) / (

        df[col].max() - df[col].min()

    )

In [6]:
df["competition_score"] = (

    df["competition_score"]

    / df["competition_score"].max()

) * 100

In [7]:
print(
    df["competition_score"]
    .describe()
)

count    50000.000000
mean        11.115152
std          7.360593
min          0.047086
25%          5.467791
50%         10.860661
75%         16.287176
max        100.000000
Name: competition_score, dtype: float64


In [8]:
def competition_label(score):

    if score < 20:
        return "Low"

    elif score < 50:
        return "Medium"

    else:
        return "High"


df["competition_label"] = (
    df["competition_score"]
    .apply(competition_label)
)

print(
    df["competition_label"]
    .value_counts()
)

competition_label
Low       46024
Medium     3750
High        226
Name: count, dtype: int64


In [9]:
features = [

    "customer_count",

    "website_traffic",

    "linkedin_followers",

    "twitter_followers",

    "media_mentions",

    "domain_authority",

    "market_size",

    "customer_segment_count"

]

X = df[features]

y = df["competition_label"]

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(y)

print(
    dict(
        zip(
            le.classes_,
            range(
                len(le.classes_)
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [12]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=12, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [13]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

preds = model.predict(X_test)

acc = accuracy_score(
    y_test,
    preds
)

print(
    "Accuracy:",
    round(acc*100,2),
    "%"
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 99.9 %
              precision    recall  f1-score   support

           0       0.85      0.98      0.91        45
           1       1.00      1.00      1.00      9205
           2       1.00      0.99      0.99       750

    accuracy                           1.00     10000
   macro avg       0.95      0.99      0.97     10000
weighted avg       1.00      1.00      1.00     10000



In [14]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
        model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(importance)

                  Feature  Importance
5        domain_authority    0.913867
1         website_traffic    0.017842
4          media_mentions    0.015582
0          customer_count    0.015533
3       twitter_followers    0.014686
6             market_size    0.011636
2      linkedin_followers    0.008527
7  customer_segment_count    0.002327


In [16]:
import joblib

joblib.dump(
    model,
    "../models/competition_analysis_model/competition_analysis_model.pkl"
)

joblib.dump(
    le,
    "../models/competition_analysis_model/label_encoder.pkl"
)

['../models/competition_analysis_model/label_encoder.pkl']

In [17]:
import json

metadata = {

    "model_name":
        "Competition Analysis Model",

    "algorithm":
        "Random Forest",

    "features":
        features,

    "classes":
        list(
            le.classes_
        )

}

with open(

    "../models/competition_analysis_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [18]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

In [19]:
print(df["competition_label"].value_counts())

print("Accuracy:", acc)

print(importance)

competition_label
Low       46024
Medium     3750
High        226
Name: count, dtype: int64
Accuracy: 0.999
                  Feature  Importance
5        domain_authority    0.913867
1         website_traffic    0.017842
4          media_mentions    0.015582
0          customer_count    0.015533
3       twitter_followers    0.014686
6             market_size    0.011636
2      linkedin_followers    0.008527
7  customer_segment_count    0.002327


In [20]:
for col in df.columns:
    if "description" in col.lower():
        print(col)